Question 5: Async Data Pipeline

In [1]:
# Importing pandas for data manipulation, sqlite3 for database operations, time for the decorator, aiosqlite for 
# async interactions with the database, and asyncio to carry out async operations:
import pandas as pd
import sqlite3
import time
import aiosqlite
import asyncio

Using the CSV file from Question 1, filtering the data to include only 'Copper' and 'Zinc' for the year 2020 & 2021:

Loading the Data from the provided CSV file, and preparing it for further processing for the task:

In [2]:
df = pd.read_csv("../data/MarketData.csv", skiprows=6)

df["Dates"] = pd.to_datetime(df["Dates"], dayfirst=True)

df = df.rename(columns={
    "PX_SETTLE": "Copper",
    "PX_SETTLE.1": "Aluminum",
    "PX_SETTLE.2": "Zinc",
    "PX_SETTLE.3": "Lead",
    "PX_SETTLE.4": "Tin",
    "PX_SETTLE.5": "CL_Futures"
})

Using the prepared DataFrame df for Filtering the DataFrame to extract only those data points for Copper and Zinc, for the years 2020 and 2021, and storing them into the filtered DataFrame, i.e., filtered_data:

In [3]:
filtered_data = df.filter(items=["Dates", "Copper", "Zinc"])

filtered_data["Dates"] = pd.to_datetime(filtered_data["Dates"], dayfirst=True)

filtered_data = filtered_data[(filtered_data["Dates"] >= "01/01/2020") & (filtered_data["Dates"] <= "31/12/2021")]

filtered_data

,Dates,Copper,Zinc
2608,2020-01-01,6174.0,2272.0
2609,2020-01-02,6188.0,2310.0
2610,2020-01-03,6129.5,2306.0
2611,2020-01-06,6138.5,2324.5
2612,2020-01-07,6149.0,2346.0
...,...,...,...
3126,2021-12-27,9568.0,3519.0
3127,2021-12-28,9568.0,3519.0
3128,2021-12-29,9680.5,3513.0
3129,2021-12-30,9691.5,3532.5


Calculating MACD (slow/medium/fast) and RSI for each metal historically:

Assumption: MACD and RSI have been calculated for the Metals, Copper and Zinc, for the years 2020 and 2021 only, as an extension of the previous sub-task.

Please Note: ewm() is called to leverage its ability in contributing to calculating the EMA, and by utilising its parameter "span", in order to specify the size of the rolling window as required in the calculations below. Calling mean() on it, aids in producing the required EMA for the respective calculation.

Moving Average Convergence Divergence (MACD):

The following formula has been used in the calculations of MACD in this Notebook:

![MACD Formula](../images/MACD_formula.png)

Please Note: ewm() is called to leverage its ability in contributing to calculating the EMA, and by utilising its parameter "span", in order to specify the size of the rolling window as required in the calculations below. Calling mean() on it, aids in producing the required EMA for the respective calculation.

Slow MACD:

For short-term, rolling window has been taken to be 24, and
for long-term, rolling window has been taken to be 52.
This has been done in order to gather a calculation of MACD with a lower sensitivity, that could be used for following trends on the long-term.

In [ ]:
copper_slow_short_term_EMA = filtered_data["Copper"].ewm(span=24).mean()

copper_slow_longer_term_EMA = filtered_data["Copper"].ewm(span=52).mean()

copper_slow_MACD = copper_slow_short_term_EMA - copper_slow_longer_term_EMA

In [ ]:
zinc_slow_short_term_EMA = filtered_data["Zinc"].ewm(span=24).mean()

zinc_slow_longer_term_EMA = filtered_data["Zinc"].ewm(span=52).mean()

zinc_slow_MACD = zinc_slow_short_term_EMA - zinc_slow_longer_term_EMA

Medium MACD:

For short-term, rolling window has been taken to be 12, and
for long-term, rolling window has been taken to be 26.
This has been done in order to gather a calculation of MACD with a more balanced sensitivity, that could be used for general trading.

In [ ]:
copper_medium_short_term_EMA = filtered_data["Copper"].ewm(span=12).mean()

copper_medium_longer_term_EMA = filtered_data["Copper"].ewm(span=26).mean()

copper_medium_MACD = copper_medium_short_term_EMA - copper_medium_longer_term_EMA

In [ ]:
zinc_medium_short_term_EMA = filtered_data["Zinc"].ewm(span=12).mean()

zinc_medium_longer_term_EMA = filtered_data["Zinc"].ewm(span=26).mean()

zinc_medium_MACD = zinc_medium_short_term_EMA - zinc_medium_longer_term_EMA

Fast MACD:

For short-term, rolling window has been taken to be 6, and
for long-term, rolling window has been taken to be 13.
This has been done in order to gather a calculation of MACD with a higher sensitivity, that could be used for following trends for the short-term.

In [ ]:
copper_fast_short_term_EMA = filtered_data["Copper"].ewm(span=6).mean()

copper_fast_longer_term_EMA = filtered_data["Copper"].ewm(span=13).mean()

copper_fast_MACD = copper_fast_short_term_EMA - copper_fast_longer_term_EMA

In [ ]:
zinc_fast_short_term_EMA = filtered_data["Zinc"].ewm(span=6).mean()

zinc_fast_longer_term_EMA = filtered_data["Zinc"].ewm(span=13).mean()

zinc_fast_MACD = zinc_fast_short_term_EMA - zinc_fast_longer_term_EMA

Relative Strength Index (RSI):

The following formula has been used in the calculations of RSI in this Notebook:

![RSI Formula](../images/RSI_formula.png)

Assumption: 14-day lookback period is being used, keeping in line with the theory of RSI, as it has been found to be a balance between sensitivity and smoothness.

In the calculation of the RSI, the following steps are followed:

-> Today's Day Price - Previous Day's Price is calculated for every data point by calling diff(). This is done to compute the gains and losses following this.

-> clip(lower=0) is called on the above calculated difference to keep all positive values the same and to convert any negative values to 0. This constitutes the gain.

-> clip(upper=0) is called on the above calculated difference to keep all negative value the same and to convert any positive value to 0. The result of this is then negated to have all these values be positives as at this point, I am aware that these are the losses and storing them separate to the gains. This constitutes the losses, i.e. hence, all positive values in the losses here.

-> The EMA is then calculated, with an alpha = 1 / period, which is the 14-day period due to the aforementioned assumption, as well as the adjust parameter is set to False, as this is faster and to be aligned with what is used in standard trading.

-> Following this, the above RS and RSI formulae are what are implemented and hence, calculated in computing the RSI here.

Calculation for Copper:

In [ ]:
# Produces the differences between the current day and the previous day:
copper_change = filtered_data["Copper"].diff()

# Gains and Losses are computed accordingly:
copper_gain = copper_change.clip(lower=0)
copper_loss = -copper_change.clip(upper=0)

# EMAs of the Gains and Losses are computed accordingly:
copper_avg_gain = copper_gain.ewm(alpha=1/14, adjust=False).mean()
copper_avg_loss = copper_loss.ewm(alpha=1/14, adjust=False).mean()

# RS and RSI formulae being applied accordingly:
copper_RS = copper_avg_gain / copper_avg_loss
copper_RSI = 100 - (100 / (1 + copper_RS))

Calculation for Zinc:

In [ ]:
# Produces the differences between the current day and the previous day:
zinc_change = filtered_data["Zinc"].diff()

# Gains and Losses are computed accordingly:
zinc_gain = zinc_change.clip(lower=0)
zinc_loss = -zinc_change.clip(upper=0)

# EMAs of the Gains and Losses are computed accordingly:
zinc_avg_gain = zinc_gain.ewm(alpha=1/14, adjust=False).mean()
zinc_avg_loss = zinc_loss.ewm(alpha=1/14, adjust=False).mean()

# RS and RSI formulae being applied accordingly:
zinc_RS = zinc_avg_gain / zinc_avg_loss
zinc_RSI = 100 - (100 / (1 + zinc_RS))

Using SQL inserts to populate the SQL table created in Question 2 with this generated data:

Establishing connection and setting up through the cursor variable which will be used to interact with database in this task:

In [ ]:
connection = sqlite3.connect("metals.db")
cursor = connection.cursor()

Deleting the current data from the table MetalPrices in the database, as I want to overwrite the data into this MetalPrices table through Insertions asynchronously:

In [ ]:
# Synchronous deleting of the data in the MetalPrices table, to set the MetalPrices table up for the async Insertion operations: 
cursor.execute("""
DELETE FROM MetalPrices;
""")

Appending all the computed MACD and RSI values to the filtered_data DataFrame, as the idea is to construct the filtered_data in such a way that it can be directly used to feed into and write into the database table:

In [ ]:
filtered_data["Copper_Slow_MACD"] = copper_slow_MACD
filtered_data["Copper_Medium_MACD"] = copper_medium_MACD
filtered_data["Copper_Fast_MACD"] = copper_fast_MACD

filtered_data["Zinc_Slow_MACD"] = zinc_slow_MACD
filtered_data["Zinc_Medium_MACD"] = zinc_medium_MACD
filtered_data["Zinc_Fast_MACD"] = zinc_fast_MACD

filtered_data["Copper_RSI"] = copper_RSI
filtered_data["Zinc_RSI"] = zinc_RSI

Demonstrating the use of a decorator to log the execution of the SQL inserts:

Modifying Question 3 to write data to the database asynchronously:

Definition of the Decorator:

In [ ]:
def log_execution(func):

    async def async_wrapper(dataframe):
        
        # Recording the time at which the database query starts:
        start = time.time()
        print("Starting DB query:")

        # Calling the passed async function, i.e., the function that will perform the respective database operation here:
        result = await func(dataframe)

        # Recording the time at which the database query finishes:
        end = time.time()
        print(f"\n\nDB query finished in {end - start:.2f} seconds\n\n")

        return result

    return async_wrapper

Function to use the above Decorator:

In [ ]:
# Function to perform the async database Writes, and the decorator to log the execution of these SQL inserts 
# has been applied. This function is an async coroutine function which enables what it is defined to do to run concurrently
# in the background:
@log_execution
async def async_write_data(filtered_data):

    # Converting the Dates column's data type back to string as SQL query can handle string data types, and not date objects
    # as this will be used for database Insertions:
    filtered_data["Dates"] = filtered_data["Dates"].astype(str)

    # Converts the data in the filtered_data DataFrame into a list of tuples, as tuples are the iterable that can be 
    # used in SQL Insertion query for SQL to execute them: 
    rows = list(filtered_data[[
        "Dates",
        "Copper",
        "Zinc",
        "Copper_Slow_MACD",
        "Copper_Medium_MACD",
        "Copper_Fast_MACD",
        "Zinc_Slow_MACD",
        "Zinc_Medium_MACD",
        "Zinc_Fast_MACD",
        "Copper_RSI",
        "Zinc_RSI"
    ]].itertuples(index=False, name=None))


    # Async connection to the database, timeout is set to 10 to allow to wait for 10 seconds if the database might unlock, 
    # if it is found to be locked at that point of time:
    async with aiosqlite.connect("metals.db", timeout=10) as db:

        await db.execute("PRAGMA journal_mode=WAL;")

        # executemany() executes the SQL query specified within it for multiple rows of data. Here, that is Insertion query, 
        # by using the list of tuples which is called rows and constructed above:
        await db.executemany("""
            INSERT INTO MetalPrices (
                Dates, Copper, Zinc,
                Copper_Slow_MACD, Copper_Medium_MACD, Copper_Fast_MACD,
                Zinc_Slow_MACD, Zinc_Medium_MACD, Zinc_Fast_MACD,
                Copper_RSI, Zinc_RSI
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, rows)

        # To permanently save the changes made to the database, the commit() is called asynchronously:
        await db.commit()

Running the write operation asynchronously:

In [17]:
# Starts the task which is in the function call, schedules for this task to run on the background, 
# i.e., concurrently while the operations on the foreground run as well:
task = asyncio.create_task(async_write_data(filtered_data))

# While the async Insertion operations are happening on the background, the following also starts running on the foreground
# simultaneously:
copper_mean = filtered_data["Copper"].mean()
zinc_mean = filtered_data["Zinc"].mean()
print("Generating analysis...")
print(f"Average Copper Price: {copper_mean:.2f}")
print(f"Average Zinc Price: {zinc_mean:.2f}")

# Now, the foreground operations have completed, and now it is waited until the background operation, i.e., the async Insertion 
# operations are completed:
await task

Generating analysis...
Average Copper Price: 7740.68
Average Zinc Price: 2643.34
Starting DB query:


OperationalError: database is locked

This error has been demonstrated here to exhibit that upon running the code to write the data asynchronously, the database was locked after some time, perhaps due to some other reason.

Due to the above error and the writes not being completed, in addition to having deleted the data in the MetalPrices data in the database prior to this for this task, I run to_sql() to re-populate the MetalPrices Table in the database with the data, so that I can proceed with the async read operations:

In [18]:
# to_sql() writes into the table specified, i.e., here, Metalprices:
filtered_data.to_sql("MetalPrices", connection, if_exists="replace", index=False)

523

Reading from the database 5 times concurruntly using async:

In [19]:
# Function to perform the async database Reads, and the decorator to log the execution of these SQL retrievals 
# has been applied. This function is an async coroutine function which enables multiple reads to run concurrently:
@log_execution
async def read_data(run_id):

    # Async connection to the database, timeout is set to 10 to allow to wait for 10 seconds if the database might unlock, 
    # if it is found to be locked at that point of time:
    async with aiosqlite.connect("metals.db", timeout=10) as db:

        print(f"[READ {run_id}] Starting read query:")

        # Executes the Retrieval query:
        async with db.execute("SELECT * FROM MetalPrices") as cursor:

            # All the query results that are hence received here as a consequence of performing a read operation, 
            # are stored in Python, in the rows variable here:
            rows = await cursor.fetchall()

            print(f"\nREAD {run_id}:\n")
            print(f"First 5 rows fetched from this Read:")
            print(rows[:5])

            print(f"This Read has now Fetched {len(rows)} rows")

            return rows

In [20]:
# gather() schedules all of these async functions to run concurrently, while the await keyword makes sure to pause at this 
# juncture until all of these async functions within gather() have been completed: 
results = await asyncio.gather(

    # read_data() creates a coroutine object for each of these calls, which are differentiated by their respective run IDs:

    read_data(1),
    read_data(2),
    read_data(3),
    read_data(4),
    read_data(5)
)

print("5 concurrent reads completed")

Starting DB query:
Starting DB query:
Starting DB query:
Starting DB query:
Starting DB query:
[READ 1] Starting read query:
[READ 2] Starting read query:
[READ 3] Starting read query:
[READ 4] Starting read query:
[READ 5] Starting read query:

READ 3:

First 5 rows fetched from this Read:
[('2020-01-01', 6174.0, 2272.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, None, None), ('2020-01-02', 6188.0, 2310.0, 0.15705128205081564, 0.3141025641016313, 0.6282051282050816, 0.4262820512817598, 0.8525641025635196, 1.7051282051279486, 100.0, 100.0), ('2020-01-03', 6129.5, 2306.0, -0.6869506679695405, -1.413852487659824, -2.9698042331856414, 0.49531141852048677, 0.9622961287659564, 1.8011991620314802, 75.67567567567568, 99.19678714859438), ('2020-01-06', 6138.5, 2324.5, -0.925829457592954, -1.8401688124531574, -3.5643400187518637, 0.8461054994982078, 1.6565564241659558, 3.139488300517769, 76.61798616448885, 99.22768453883856), ('2020-01-07', 6149.0, 2346.0, -0.8548799485606651, -1.615263376207622, -2.7672892

Therefore, from the above output of the print statements due to where the print statements have been placed, it can be verified that the read operations have occurred 5 times concurruntly using async.

References:

-> https://b2broker.com/news/macd-indicator-how-to-read-the-chart/

->https://www.alpharithms.com/relative-strength-index-rsi-in-python-470209/#google_vignette